In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -U langchain langchain-community langchain-text-splitters pypdf chromadb sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\user\AppData\Local\Temp\ipykernel_1048\1866228256.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
PDF_FOLDER = "documents"

In [5]:
documents = []

for file in os.listdir(PDF_FOLDER):

    if file.endswith(".pdf"):

        pdf_path = os.path.join(PDF_FOLDER, file)

        loader = PyPDFLoader(pdf_path)

        docs = loader.load()

        documents.extend(docs)

        print(f"Loaded: {file} | Pages: {len(docs)}")

Loaded: 243-02WH-17308.pdf | Pages: 28
Loaded: 9789241562638_eng.pdf | Pages: 108
Loaded: npwdr_complete_table.pdf | Pages: 7
Loaded: trs970-annex2-gmp-wate-pharmaceutical-use.pdf | Pages: 23


In [6]:
print(f"Total Pages = {len(documents)}")

Total Pages = 166


In [7]:
documents[0]

Document(metadata={'producer': '', 'creator': 'ABBYY FineReader', 'creationdate': 'D:20080710212710', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': 'D:20080710212710', 'source': 'documents\\243-02WH-17308.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}, page_content="World Health Organization\nSustainable Development and\nHealthy Environments\nistr. i,.•,.,•„;•: .\nI'X >••] •!\nWHO/SDE/WSH/02.02\nDistr.: Limited\nEnglish only\n••. ' i,r ': '\n'•' '-I ;' i\nWHO Guidelines for Drinking\nWater Quality\nPolicies and Procedures for Preparing\nand Updating of the WHO Guidelines\nfor Drinking-Water Quality\nVersion at January 2002\nProtection of the Human Environment\nWater, Sanitation and Health\nGeneva, 2002")

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [9]:
chunks = text_splitter.split_documents(documents)

In [10]:
print(f"Number of Chunks = {len(chunks)}")

Number of Chunks = 493


In [11]:
chunks[0]

Document(metadata={'producer': '', 'creator': 'ABBYY FineReader', 'creationdate': 'D:20080710212710', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': 'D:20080710212710', 'source': 'documents\\243-02WH-17308.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}, page_content="World Health Organization\nSustainable Development and\nHealthy Environments\nistr. i,.•,.,•„;•: .\nI'X >••] •!\nWHO/SDE/WSH/02.02\nDistr.: Limited\nEnglish only\n••. ' i,r ': '\n'•' '-I ;' i\nWHO Guidelines for Drinking\nWater Quality\nPolicies and Procedures for Preparing\nand Updating of the WHO Guidelines\nfor Drinking-Water Quality\nVersion at January 2002\nProtection of the Human Environment\nWater, Sanitation and Health\nGeneva, 2002")

In [12]:
!pip install -U langchain-huggingface


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
from langchain_community.vectorstores import Chroma

In [16]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="vector_db"
)

In [17]:
retriever = vector_db.as_retriever()

In [18]:
results = retriever.invoke(
    "What is the acceptable pH for drinking water?"
)

In [19]:
print(results[0].page_content)

Copper 1.0 mg/L
Corrosivity Noncorrosive
Fluoride 2.0 mg/L
Foaming Agents 0.5 mg/L
Iron 0.3 mg/L
Manganese 0.05 mg/L
Odor 3 threshold odor number
pH 6.5-8.5
Silver 0.10 mg/L
Sulfate 250 mg/L
Total Dissolved Solids 500 mg/L
Zinc 5 mg/L
visit: epa.gov/safewater
call: (800) 426-4791
FOR MORE INFORMATION ON EPA’S  
SAFE DRINKING WATER:
ADDITIONAL INFORMATION:
National Primary Drinking Water Regulations EPA 816-F-09-004   |   MAY 2009


In [20]:
question = "What is the acceptable pH for drinking water?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)


prompt = f"""
You are a water quality assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

# LLM Model

In [21]:
!pip install -q langchain-groq


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
!pip install -q langchain-classic


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
from getpass import getpass

# مجاني بالكامل - سجل حساب على https://console.groq.com واحصل على API key من هناك
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

## تخصيص الـ Prompt (يسمح للموديل يجاوب من معرفته العامة لو المعلومة مش في المستندات)


In [25]:
from langchain_core.prompts import PromptTemplate

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""أنت مساعد ذكي. استخدم المعلومات التالية للمساعدة في الإجابة إن كانت متاحة،
لكن لو المعلومات المتاحة لا تغطي السؤال، أجب من معرفتك العامة بشكل طبيعي بدون ما تقول "لا أعرف".

المعلومات المتاحة:
{context}

السؤال:
{question}

الإجابة:"""
)

In [26]:
from langchain_classic.chains import RetrievalQA


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": custom_prompt}
)

## اختبار النظام (Test the RAG chain)

In [27]:
question = "What is the acceptable pH for drinking water?"

result = qa_chain.invoke({"query": question})

print("Question:", question)
print("\nAnswer:", result["result"])

Question: What is the acceptable pH for drinking water?

Answer: الأس هيدروجيني (pH) المقبول للمياه الشرب هو بين 6.5 و 8.5.


## عرض المصادر (Source documents) - اختياري

In [28]:
# لعرض المصادر، يجب إعادة إنشاء الـ chain مع return_source_documents=True
qa_chain_with_sources = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": custom_prompt},
    return_source_documents=True,
)

result = qa_chain_with_sources.invoke({"query": question})

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "?")
    print(f"{i}. {source} - page {page}")
    print(doc.page_content[:200], "...\n")

Answer: الأس هيدروجيني (pH) المقبول للمياه الشرب هو بين 6.5 و 8.5.

--- Sources ---
1. documents\npwdr_complete_table.pdf - page 6
Copper 1.0 mg/L
Corrosivity Noncorrosive
Fluoride 2.0 mg/L
Foaming Agents 0.5 mg/L
Iron 0.3 mg/L
Manganese 0.05 mg/L
Odor 3 threshold odor number
pH 6.5-8.5
Silver 0.10 mg/L
Sulfate 250 mg/L
Total Dis ...

2. documents\npwdr_complete_table.pdf - page 6
Copper 1.0 mg/L
Corrosivity Noncorrosive
Fluoride 2.0 mg/L
Foaming Agents 0.5 mg/L
Iron 0.3 mg/L
Manganese 0.05 mg/L
Odor 3 threshold odor number
pH 6.5-8.5
Silver 0.10 mg/L
Sulfate 250 mg/L
Total Dis ...

3. documents\npwdr_complete_table.pdf - page 6
Copper 1.0 mg/L
Corrosivity Noncorrosive
Fluoride 2.0 mg/L
Foaming Agents 0.5 mg/L
Iron 0.3 mg/L
Manganese 0.05 mg/L
Odor 3 threshold odor number
pH 6.5-8.5
Silver 0.10 mg/L
Sulfate 250 mg/L
Total Dis ...

4. documents\npwdr_complete_table.pdf - page 6
Copper 1.0 mg/L
Corrosivity Noncorrosive
Fluoride 2.0 mg/L
Foaming Agents 0.5 mg/L
Iron 0.3 mg/L
Manganese 0.0

## دالة جاهزة لطرح أي سؤال (Ask anything)

In [29]:
def ask(question: str):
    result = qa_chain_with_sources.invoke({"query": question})
    print("Q:", question)
    print("A:", result["result"])
    print("\nSources:")
    for doc in result["source_documents"]:
        print(" -", doc.metadata.get("source", "unknown"), "| page", doc.metadata.get("page", "?"))
    return result

In [30]:
ask("What are the main sources of water pollution?")

Q: What are the main sources of water pollution?
A: uspended solids 
colloids 
dissolved solids 
nutrients 
bacteria 
viruses 
parasites 
algae 
inorganic compounds 
metals 
asbestos 
volatile organic compounds 
radioactive materials 
pesticides 
turbidity 
foaming agents 
taste 
odour 
disinfection byproducts 
color 
pH 
temperature 
conductivity 
hardness 
alkalinity 
total dissolved solids 
solids 
residual chlorine 
other parameters 
the following are some of the most common water quality parameters:
suspended solids 
colloids 
dissolved solids 
nutrients 
bacteria 
viruses 
parasites 
algae 
inorganic compounds 
metals 
asbestos 
volatile organic compounds 
radioactive materials 
pesticides 
turbidity 
foaming agents 
taste 
odour 
disinfection byproducts 
color 
pH 
temperature 
conductivity 
hardness 
alkalinity 
total dissolved solids 
solids 
residual chlorine 
other parameters 
the following are some of the most common water quality parameters:
suspended solids 
colloids 
dis

{'query': 'What are the main sources of water pollution?',
 'result': 'uspended solids \ncolloids \ndissolved solids \nnutrients \nbacteria \nviruses \nparasites \nalgae \ninorganic compounds \nmetals \nasbestos \nvolatile organic compounds \nradioactive materials \npesticides \nturbidity \nfoaming agents \ntaste \nodour \ndisinfection byproducts \ncolor \npH \ntemperature \nconductivity \nhardness \nalkalinity \ntotal dissolved solids \nsolids \nresidual chlorine \nother parameters \nthe following are some of the most common water quality parameters:\nsuspended solids \ncolloids \ndissolved solids \nnutrients \nbacteria \nviruses \nparasites \nalgae \ninorganic compounds \nmetals \nasbestos \nvolatile organic compounds \nradioactive materials \npesticides \nturbidity \nfoaming agents \ntaste \nodour \ndisinfection byproducts \ncolor \npH \ntemperature \nconductivity \nhardness \nalkalinity \ntotal dissolved solids \nsolids \nresidual chlorine \nother parameters \nthe following are som